# Vector Stores

Once embeddings are created, we need to store them in a way that allows us to calculate semantic similarity. So simply saving the embeddings in a CSV file is not enough. We need to use specialised vector stores and databases.

#### Setup

Install the required libraries, chunk documents, and load the saved embeddings file `"rag_embeddings.csv"`

In [14]:
# install the vector DBs
# %pip install chromadb faiss-cpu qdrant-client

In [15]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [16]:
dir_path = "data/Policy Documents"

loader = DirectoryLoader(
    dir_path,
    glob="*.pdf",
    loader_cls=PyPDFLoader
)

documents = loader.load()

# Recursive character splitting
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks = recursive_splitter.split_documents(documents)

In [17]:
chunks[0]

Document(metadata={'producer': 'Microsoft® Office Word 2007', 'creator': 'Microsoft® Office Word 2007', 'creationdate': '2023-08-24T19:47:11+05:30', 'title': 'Exide Life Group Term Life (UIN 114N012V03) – Terms and Conditions', 'author': 'Atul Bhatia', 'moddate': '2023-08-24T19:47:11+05:30', 'source': 'data/Policy Documents/HDFC-Life-Group-Term-Life-Policy.pdf', 'total_pages': 30, 'page': 0, 'page_label': '1'}, page_content='F&U dated 15th October 2022                  UIN-101N169V02  P a g e  | 0                        \n \n \n \n \n \n   HDFC Life Group Term Life \n \nOF \n \n \n«OWNERNAME» \n \n \n \n \n \n  \nBased on the Proposal and the declarations and \nany \nstatement made or referred to therein,')

In [18]:
import pandas as pd, numpy as np
embeddings_df = pd.read_csv("data/embeddings/rag_embeddings.csv")

Convert the pandas DF back to original type of the embeddings, numpy array

In [19]:
embeddings = np.array(embeddings_df).astype("float32")

---
## ChromaDB

Once we have embeddings, we need somewhere to **store** them and **search** them efficiently. This is the job of a vector store (or vector database).

**ChromaDB** is an open-source, developer-friendly vector database designed for simplicity. Its standout features include:

- **Built-in embedding**: Pass raw text and ChromaDB embeds it for you (using `all-MiniLM-L6-v2` by default). No manual embedding step needed.
- **Metadata filtering**: Attach key-value metadata to each document and filter on it at query time. For example, retrieve only documents from a specific source or date range.
- **Persistent storage**: Use `PersistentClient` to save data to disk so it survives process restarts.
- **Zero infrastructure**: Runs in-process, no separate server to manage.

[Chroma Documentation](https://docs.trychroma.com/docs/overview/introduction)

ChromaDB organises vectors into **collections** (analogous to tables in a relational DB). Below we create a collection, add documents with metadata, and run a semantic query.

In [20]:
import chromadb
# initialise client
chroma_client = chromadb.Client()

To fetch or create a collection with the given name and metadata, use the `get_or_create_collection()` method. If the collection already exists, the metadata provided is ignored. If the collection does not exist, the new collection will be created with the provided metadata. 

`embedding_function`: Optional function to use to embed documents

In [ ]:
chroma_collection = chroma_client.get_or_create_collection(
    name="demo_collection",
    metadata={"hnsw:space": "cosine"} # Hierarchical Navigable Small World.
    # embedding_function=<DefaultEmbeddingFunction>   ← implicit
)

We can either add the text chunks directly and let Chroma embed them internally, or directly use the `embeddings` parameter instead of `documents`

In [22]:
texts = [doc.page_content for doc in chunks]
metadatas = [doc.metadata for doc in chunks]

chroma_collection.add(
    documents=texts,
    metadatas=metadatas,
    ids=[f"doc_{i}" for i in range(len(texts))]
)

print(f"Collection '{chroma_collection.name}' created with {chroma_collection.count()} documents")

Collection 'demo_collection' created with 2373 documents


To search the embeddings corpus for a query, we use the `.query()` method to get the `n_results` nearest neighbor embeddings for provided `query_embeddings` (user queries already transformed into embeddings) or `query_texts` (textual queries which are first converted to vectors).

In [33]:
query = "what is the policy on eye issues?"

chroma_results = chroma_collection.query(
    query_texts=[query],
    n_results=3,     # top 3 results
    include=["documents", "metadatas", "distances", "embeddings"],

)
chroma_results

{'ids': [['doc_523', 'doc_423', 'doc_1669']],
 'embeddings': [array([[-0.05163161,  0.03799706,  0.01301469, ...,  0.05714218,
           0.01807904, -0.0339192 ],
         [ 0.0704445 ,  0.01832258, -0.00429443, ...,  0.01541501,
          -0.09451009, -0.0152937 ],
         [ 0.04259059,  0.01026398, -0.01197558, ...,  0.02960346,
          -0.07551827, -0.02124671]])],
 'documents': [['7. Routine eye tests, any Dental Treatment or Surgery of cosmetic nature, extraction of impacted \ntooth/teeth, orthodontics or orthognathic surgery, or tempero-mandibular joint disorder except as \nnecessitated by an accidental injury and warranting Hospitalization; \n8. Outpatient treatment;',
   '12 Blindness \nTotal, permanent and irreversible loss of all vision in both eyes as a result of illness or \naccident. \nThe Blindness is evidenced by: \ni. corrected visual acuity being 3/60 or less in both eyes or ; \nii. the field of vision being less than 10 degrees in both eyes.',
   '12. Blindness- T

As you can see, it contains several fields. The important ones are: 'metadatas', 'documents', and 'distances'.

In [24]:
chroma_results["documents"], chroma_results["distances"]

([['7. Routine eye tests, any Dental Treatment or Surgery of cosmetic nature, extraction of impacted \ntooth/teeth, orthodontics or orthognathic surgery, or tempero-mandibular joint disorder except as \nnecessitated by an accidental injury and warranting Hospitalization; \n8. Outpatient treatment;',
   '12 Blindness \nTotal, permanent and irreversible loss of all vision in both eyes as a result of illness or \naccident. \nThe Blindness is evidenced by: \ni. corrected visual acuity being 3/60 or less in both eyes or ; \nii. the field of vision being less than 10 degrees in both eyes.',
   '12. Blindness- Total, permanent and irreversible loss of all vision in both eyes as a result of illness or accident. \nThe Blindness is evidenced by: \n\uf0b7 corrected visual acuity being 3/60 or less in both eyes or ; \n\uf0b7 the field of vision being less than 10 degrees in both eyes.']],
 [[0.4837631583213806, 0.5137187838554382, 0.5195977687835693]])

In [25]:
chroma_results["metadatas"]

[[{'moddate': '2021-12-09T06:23:28+00:00',
   'author': 'ANINDYAA',
   'title': 'HDFC Life Easy Health - 101N110V03 - Policy Bond (Single Pay)',
   'creationdate': '2021-11-29T10:03:02+00:00',
   'total_pages': 33,
   'producer': 'Microsoft: Print To PDF',
   'source': 'data/Policy Documents/HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf',
   'page': 16,
   'creator': 'PyPDF',
   'page_label': '17'},
  {'title': 'HDFC Life Easy Health - 101N110V03 - Policy Bond (Single Pay)',
   'creator': 'PyPDF',
   'producer': 'Microsoft: Print To PDF',
   'total_pages': 33,
   'page_label': '7',
   'author': 'ANINDYAA',
   'creationdate': '2021-11-29T10:03:02+00:00',
   'source': 'data/Policy Documents/HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf',
   'moddate': '2021-12-09T06:23:28+00:00',
   'page': 6},
  {'page': 27,
   'producer': 'Microsoft: Print To PDF',
   'creator': 'PyPDF',
   'title': 'HDFC Life Group Poorna Suraksha (101N137V02) - Policy Document',
   'creati

---
## FAISS

**FAISS** (Facebook AI Similarity Search) is a C++ library (with Python bindings) developed by Meta Research for efficient similarity search over dense vectors. Unlike ChromaDB, FAISS is **not a database**. It is a low-level indexing library. This means:

- **You manage embeddings yourself**: FAISS stores only the numerical vectors; you maintain the mapping from vector index to document text externally
- **No built-in metadata filtering**: Filtering must be done post-retrieval or with a wrapper
- **Extreme speed**: FAISS is optimised for performance with support for GPU acceleration, product quantisation, and inverted file (IVF) indices that scale to billions of vectors

Although, we are using `faiss-cpu`. You can use the GPU powered variant as well. It is however used with CUDA and might need a more involved installation.

FAISS is the right choice when raw retrieval speed is the priority. For example, in research experiments or when building a custom retrieval layer. 

[FAISS Documentation](https://faiss.ai/index.html)


Like Chroma collections, we create indexes here (Chroma is more DBMS oriented, with CRUD operations). 

We use `IndexFlatIP` (flat inner-product index) here, which performs exact search. For large corpora, approximate indices like `IndexIVFFlat` or `IndexHNSWFlat` trade a small accuracy loss for dramatically faster search.

In [26]:
import faiss

dimension = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)  # Inner Product (equivalent to cosine for normalised vectors)

For inner-product similarity calculation, we also need to normalise the vectors.

Note that you can also use cosine-similarity based index, `faiss.IndexFlatL2(dimension)`. This does not need embeddings to be normalised. In fact, this uses inner product + normalised vectors, as faiss doesn't directly implement cosine similarity.

Pandas operations can produce arrays with non-contiguous strides. `faiss.normalize_L2()` requires the NumPy array to be C-contiguous (stored as a single memory block in row-major order) in memory. When data comes from pandas to numpy, the resulting array may be non-contiguous. So, we need to convert embeddings into a C-contiguous array first.

In [27]:
print(embeddings.flags)

  C_CONTIGUOUS : False
  F_CONTIGUOUS : True
  OWNDATA : True
  WRITEABLE : True
  ALIGNED : True
  WRITEBACKIFCOPY : False



In [28]:
embeddings = np.ascontiguousarray(embeddings)

In [29]:
# normalise and add embeddings to the index
faiss.normalize_L2(embeddings)
faiss_index.add(embeddings)

print(f"FAISS index created")
print(f"  Dimension  : {dimension}")
print(f"  Vectors    : {faiss_index.ntotal}")
print(f"  Metric     : Inner Product (cosine on normalised vectors)")

FAISS index created
  Dimension  : 384
  Vectors    : 2373
  Metric     : Inner Product (cosine on normalised vectors)


Now, we need to also embed the user query as vectors to enable semantic search

In [30]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

query_embedding = embedding_model.encode([query])
faiss.normalize_L2(query_embedding)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Here, we use the `.search()` method to search the index against user query

In [31]:
scores, indices = faiss_index.search(query_embedding, k=3)  # top 3 results

print(f"Query: '{query}'\n")
print("FAISS Results:")
print("-" * 60)
for idx, score in zip(indices[0], scores[0]):
    print(f"  (score: {score:.4f}) {chunks[idx]}")

Query: 'what is the policy on eye issues?'

FAISS Results:
------------------------------------------------------------
  (score: 0.5162) page_content='Member under this Policy unless it receives the information from the Policy Holder about the death of the Insured 
Member within 30 days of the occurrence of that event and unless the Claim is in the prescribed form' metadata={'producer': 'Microsoft® Office Word 2007', 'creator': 'Microsoft® Office Word 2007', 'creationdate': '2023-08-24T19:47:11+05:30', 'title': 'Exide Life Group Term Life (UIN 114N012V03) – Terms and Conditions', 'author': 'Atul Bhatia', 'moddate': '2023-08-24T19:47:11+05:30', 'source': 'data/Policy Documents/HDFC-Life-Group-Term-Life-Policy.pdf', 'total_pages': 30, 'page': 15, 'page_label': '16'}
  (score: 0.4863) page_content='Employer. 
 
18. Company means HDFC Life Insurance Company Limited. 
 
19. Coverage Schedule shall mean Coverage Schedule appended to this Policy giving the details of 
the Insured Members cov

---
## Qdrant

**Qdrant** (pronounced "quadrant") is a production-grade, open-source vector database written in Rust. It sits between ChromaDB's simplicity and a fully managed cloud service like Pinecone:

- **Rich payload filtering**: Attach arbitrary JSON payloads to each vector and filter on them during search using conditions like `$eq`, `$in`, `$gt`, range filters, and even geo-filters. Filters are applied *during* the ANN search (not post-retrieval), so they don't degrade performance.
- **Multiple deployment modes**: In-memory (for prototyping, as we use here), single-node file-based, or distributed cluster for production
- **Batch and streaming upserts**: Efficiently ingest large volumes of vectors
- **Named vectors**: Store multiple vector representations per point (e.g., title embedding + body embedding)


[Documentation](https://qdrant.tech/documentation/overview/) and [API Reference](https://api.qdrant.tech/api-reference)


Qdrant requires you to bring your own embeddings, similar to FAISS, but provides full database semantics, such as persistence, collections, CRUD, and filtering, like Chroma.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

qdrant_client = QdrantClient(":memory:")    # create a client using in-memory instrace, without a server; takes the URL for cloud/local servers

Qdrant also uses collections. We can create a collection using `create_collection()` method. It also takes the collection name and vector configuration (dimensions, distance calculation method)

In [42]:
qdrant_client.create_collection(
    collection_name="demo_collection",
    vectors_config=VectorParams(size=dimension, distance=Distance.COSINE)
)

True

In Qdrant, data entries are called points. A point contains three components, `id` (unique identifier), `vector` (embedding vector), and `payload` (metadata).

In [52]:
from qdrant_client.models import PointStruct

points = []

for i, (text, emb) in enumerate(zip(chunks, embeddings)):
    points.append(PointStruct(
                                id=i,
                                vector=emb.tolist(),
                                payload = {
                                    "text": text.page_content,
                                    "metadata": text.metadata,
                                    "source": text.metadata['source'],
                                    "page": text.metadata['page']
                                    }
                            ))

For inserting points, Qdrant uses `upsert` (update or insert)

In [53]:
qdrant_client.upsert(collection_name="demo_collection", points=points)

print(f"Qdrant collection created with {len(points)} points")

Qdrant collection created with 2373 points


Finally, to query a collection, we generate a query embedding and use the `query_points()` method

In [55]:
query_vec = embedding_model.encode(query).tolist()

qdrant_results = qdrant_client.query_points(
    collection_name="demo_collection",
    query=query_vec,
    limit=3
)

In [59]:
qdrant_results

QueryResponse(points=[ScoredPoint(id=166, version=0, score=0.516236608495825, payload={'text': '7. Routine eye tests, any Dental Treatment or Surgery of cosmetic nature, extraction of impacted \ntooth/teeth, orthodontics or orthognathic surgery, or tempero-mandibular joint disorder except as \nnecessitated by an accidental injury and warranting Hospitalization; \n8. Outpatient treatment;', 'metadata': {'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2021-11-29T10:03:02+00:00', 'author': 'ANINDYAA', 'moddate': '2021-12-09T06:23:28+00:00', 'title': 'HDFC Life Easy Health - 101N110V03 - Policy Bond (Single Pay)', 'source': 'data\\Policy Documents\\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf', 'total_pages': 33, 'page': 16, 'page_label': '17'}, 'source': 'data\\Policy Documents\\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf', 'page': 16}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=66, version=0, score=0.486281278

In [60]:
qdrant_results.points

[ScoredPoint(id=166, version=0, score=0.516236608495825, payload={'text': '7. Routine eye tests, any Dental Treatment or Surgery of cosmetic nature, extraction of impacted \ntooth/teeth, orthodontics or orthognathic surgery, or tempero-mandibular joint disorder except as \nnecessitated by an accidental injury and warranting Hospitalization; \n8. Outpatient treatment;', 'metadata': {'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2021-11-29T10:03:02+00:00', 'author': 'ANINDYAA', 'moddate': '2021-12-09T06:23:28+00:00', 'title': 'HDFC Life Easy Health - 101N110V03 - Policy Bond (Single Pay)', 'source': 'data\\Policy Documents\\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf', 'total_pages': 33, 'page': 16, 'page_label': '17'}, 'source': 'data\\Policy Documents\\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf', 'page': 16}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=66, version=0, score=0.4862812784525548, payload={'t

## Summary

| Store | Type | Embedding | Persistence | Filtering | Best For |
|---|---|---|---|---|---|
| **ChromaDB** | Vector DB | Built-in | File-based | Metadata filters | Prototyping, small-to-medium scale |
| **FAISS** | Library | Bring Your Own | Manual (save/load) | None built-in | Raw speed, research, GPU workloads |
| **Qdrant** | Vector DB | BYO | File / server | Rich payload filters | Production, complex filtering |